# BERTopic — Template para Novo Corpus

Copie este notebook para `notebooks/XX_bertopic_<nome>.ipynb` e substitua
os placeholders marcados com `# <<` para adaptar a um novo corpus.

## Pipeline (5 etapas)
1. **Pré-processamento** — stop words customizadas, CountVectorizer com acentos
2. **Embeddings** — Qwen3 0.6B via Ollama (1024d nativo, cacheados)
3. **Redução + Clustering** — UMAP 5D → HDBSCAN (leaf)
4. **Representação** — c-TF-IDF + MMR (diversity=0.2)
5. **Pós-processamento** — reduce_outliers → update_topics → nomeação LLM

## 6 Métricas avaliadas
Topic Diversity, Taxa de outliers, Coerência C_v, Coerência Semântica,
Exclusividade, FREX (Frequency × Exclusivity)

In [ ]:
# Descomente se necessário
# !pip install bertopic gensim sentence-transformers umap-learn hdbscan httpx nest_asyncio itables wordcloud -q
# !python -m spacy download pt_core_news_lg -q

In [ ]:
SUFFIX     = bt_cfg.get("embedding_suffix", "4096d")
# Embeddings ficam em 02-embeddings/data/output/ — sem necessidade de cópia local
CACHE_PATH = Path("../../02-embeddings/data/output") / INPUT_DIR.name / f"embeddings_{SAFE_MODEL}_{SUFFIX}.npy"
print(f"Modelo   : {MODEL_NAME}  |  Dim: {DIMENSION or 'nativo 4096d'}  |  Backend: {BACKEND}")

In [ ]:
df   = load_corpus(INPUT_DIR)
docs = df[TEXT_COL].astype(str).tolist()
post_ids = df[POST_ID_COL].astype(str).tolist()

t0 = time.time()
embeddings = get_or_compute_embeddings(
    texts=docs,
    model_name=MODEL_NAME,
    cache_path=str(CACHE_PATH),
    backend=BACKEND,
    dimension=DIMENSION,
    show_progress=True,
)
assert embeddings.shape[0] == len(docs), "Desalinhamento corpus ↔ embeddings"

print(f"Docs       : {len(docs)}")
print(f"Embeddings : {embeddings.shape}  (tempo: {time.time()-t0:.0f}s)")
print(f"\nExemplo: {docs[0][:120]}")

## Pré-processamento — CountVectorizer

O CountVectorizer do BERTopic gera as keywords (c-TF-IDF) — **não** os embeddings.
Os três ajustes críticos:

1. **Stop words customizadas** — NLTK base + termos de domínio específicos do corpus
2. **ngram_range=(1,2)** — habilita bigramas relevantes (ex: "saúde pública")
3. **token_pattern** — preserva acentos PT-BR (`À-ÿ`) e tokens de emojis demojizados (`_`)

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords", quiet=True)

# Stop words base por idioma
if LANGUAGE == "pt":
    base_sw = set(stopwords.words("portuguese"))
else:
    base_sw = set(sk_text.ENGLISH_STOP_WORDS)

# << Adicione stop words de domínio do corpus abaixo
STOPWORDS_DOMAIN = set([
    # Ex para social: "raquellyra", "governodepernambuco", "pernambuco"
])

# Stop words de emojis demojizados — vêm de params.yaml > corpora.<corpus>.stopwords_emojis
STOPWORDS_EMOJIS = set(corpus_cfg.get("stopwords_emojis", []))

all_stop = base_sw | STOPWORDS_DOMAIN | STOPWORDS_EMOJIS

vectorizer = CountVectorizer(
    stop_words=list(all_stop),
    ngram_range=tuple(bt_cfg["vectorizer"]["ngram_range"]),
    token_pattern=TOKEN_PATTERN,
    min_df=2,
)
print(f"Stop words total: {len(all_stop)}")

In [ ]:
umap_p = bt_cfg["umap"]
umap_model = UMAP(
    n_components=umap_p["n_components"],
    n_neighbors=umap_p["n_neighbors"],
    min_dist=umap_p["min_dist"],
    metric=umap_p["metric"],
    random_state=seed,
)

hdbscan_p = bt_cfg["hdbscan"]
hdbscan_model = HDBSCAN(
    min_cluster_size=hdbscan_p["min_cluster_size"],
    min_samples=hdbscan_p["min_samples"],
    cluster_selection_method=hdbscan_p["cluster_selection_method"],
    prediction_data=True,
)

representation_model = MaximalMarginalRelevance(diversity=bt_cfg["mmr_diversity"])

topic_model = BERTopic(
    language=BT_LANGUAGE,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    representation_model=representation_model,
    min_topic_size=bt_cfg["min_topic_size"],
    calculate_probabilities=True,
    verbose=True,
)

t0 = time.time()
topics, probs = topic_model.fit_transform(docs, embeddings)
print(f"\nTreinamento: {time.time()-t0:.1f}s")
print(f"Tópicos encontrados: {topic_model.get_topic_info().shape[0] - 1} (+1 outlier)")
print(f"Outliers (-1): {sum(t == -1 for t in topics)} ({sum(t == -1 for t in topics)/len(topics):.1%})")

In [ ]:
# ORDEM CORRETA (B18):
#   reduce_outliers  → retorna novos assignments (NÃO atualiza o modelo)
#   update_topics    → sincroniza o modelo com os assignments sem outliers  ← obrigatório antes de reduce_topics
#   reduce_topics    → mescla os menores tópicos até nr_topics
#   update_topics    → atualiza representações c-TF-IDF + MMR após a redução
# Pular o 1º update_topics faz reduce_topics operar sobre assignments desatualizados.

# 1. Reatribuir outliers via c-TF-IDF
new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")

# 2. Sincronizar modelo com assignments sem outliers
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer,
                           representation_model=representation_model)

# 3. Reduzir para K tópicos alvo
topic_model.reduce_topics(docs, nr_topics=bt_cfg["reduce_topics_nr"])
new_topics = topic_model.topics_

# 4. Atualizar representações c-TF-IDF + MMR com K final
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer,
                           representation_model=representation_model)
topics = new_topics
topic_info = topic_model.get_topic_info()

print(f"Tópicos após redução: {topic_info[topic_info.Topic != -1].shape[0]}")
print(f"Outliers remanescentes: {sum(t == -1 for t in topics)}")
topic_info.head(10)

In [ ]:
llm_cfg = params.get("llm", {})
valid_ids = [t for t in topic_info["Topic"].tolist() if t != -1]
top_n_llm = params["evaluation"].get("top_n_keywords_llm", 10)

topics_keywords = {
    tid: [w for w, _ in topic_model.get_topic(tid)[:top_n_llm]]
    for tid in valid_ids
}
example_docs_map = {tid: topic_model.get_representative_docs(tid) for tid in valid_ids}

print(f"Nomeando {len(valid_ids)} tópicos via {llm_cfg.get('model')} (lang={LANGUAGE})...")
t0 = time.time()
try:
    rotulos = name_all_topics(
        topics_keywords,
        model=llm_cfg["model"],
        base_url=llm_cfg.get("base_url", "http://localhost:11434"),
        example_docs_map=example_docs_map,
        lang=LANGUAGE,
    )
    print(f"done em {time.time()-t0:.0f}s")
except Exception as e:
    print(f"LLM falhou ({e}) — fallback top-3 keywords")
    rotulos = {tid: ", ".join(kws[:3]) for tid, kws in topics_keywords.items()}

for tid in sorted(rotulos):
    print(f"  T{tid}: {rotulos[tid]}")

topic_model.set_topic_labels(rotulos)

In [ ]:
top_n_metrics = params["evaluation"].get("top_n_keywords_metrics", 20)
topics_keywords_metrics = {
    tid: [w for w, _ in topic_model.get_topic(tid)[:top_n_metrics]] for tid in valid_ids
}

# 1. Taxa de outliers
outlier_rate = sum(t == -1 for t in topics) / len(topics)

# 2. Topic Diversity (Dieng et al., 2020)
td_score = compute_topic_diversity(topics_keywords_metrics, top_k=top_n_metrics)

# 3. Coerência C_v via gensim (tokenização lematizada, idioma-aware)
#    Amostra de até 5000 docs (seed 42) — PADRONIZADO com os notebooks de corpus
#    (evita lematização lenta de spaCy em corpora grandes).
from _helpers import lemmatize_corpus
_sample_idx = np.random.RandomState(42).choice(len(docs), min(5000, len(docs)), replace=False)
_docs_sample = [docs[i] for i in _sample_idx]
tokenized, dictionary = lemmatize_corpus(_docs_sample, LANGUAGE, params)
cv_score = compute_coherence_cv(topics_keywords_metrics, tokenized, dictionary)

# 4. Coerência Semântica (similaridade de embeddings entre keywords)
if BACKEND == "ollama":
    from _helpers import get_ollama_embeddings
    def _embed(texts):
        return get_ollama_embeddings(list(texts), MODEL_NAME, dimension=DIMENSION)
else:
    from sentence_transformers import SentenceTransformer
    _st_model = SentenceTransformer(MODEL_NAME)
    def _embed(texts):
        return _st_model.encode(list(texts), show_progress_bar=False, convert_to_numpy=True)

sem_score = compute_semantic_coherence(topics_keywords_metrics, _embed)

# 5. Exclusividade — c-TF-IDF completo (não só top-N do próprio tópico)
ctfidf_full = topic_model.c_tf_idf_.toarray()
vocab_full  = topic_model.vectorizer_model.get_feature_names_out()
topic_idx   = {tid: i for i, tid in enumerate(sorted(topic_model.get_topics().keys()))}
topic_word_scores = {
    tid: {vocab_full[j]: float(ctfidf_full[topic_idx[tid]][j])
          for j in range(len(vocab_full)) if ctfidf_full[topic_idx[tid]][j] > 0}
    for tid in valid_ids
}
excl_score, excl_per_topic = compute_exclusivity_ctfidf(
    topics_keywords_metrics, topic_word_scores, top_n=top_n_metrics
)

# 6. FREX (Airoldi & Bischof, 2016 — harmônica freq × exclusividade)
vocab_index = {w: i for i, w in enumerate(vocab_full)}
frex_score, frex_per_topic = compute_frex_score(
    topics_keywords_metrics, topic_word_matrix=ctfidf_full,
    vocab_index=vocab_index, topic_index=topic_idx, top_n=top_n_metrics, w_freq=0.5,
)

print("=" * 55)
print(f"  Tópicos válidos      : {len(valid_ids)}")
print(f"  Taxa de outliers     : {outlier_rate:.1%}")
print(f"  Topic Diversity      : {td_score:.4f}  (↑ melhor)")
print(f"  Coerência C_v        : {cv_score:.4f}              (↑ melhor)")
print(f"  Coer. Semântica      : {sem_score:.4f}              (↑ melhor)")
print(f"  Exclusividade        : {excl_score:.4f}              (↑ melhor)")
print(f"  FREX                 : {frex_score:.4f}              (↑ melhor)")
print("=" * 55)

## Sweep de estrategias/threshold de reduce_outliers + grid UMAP/HDBSCAN

Usa `_helpers.sweep_outlier_strategies` / `sweep_outlier_threshold` /
`sweep_bertopic_grid` para comparar empiricamente, via multiplos seeds,
alternativas ao pipeline fixo acima (B18 com a estrategia/nn/mcs do
`params.yaml`). Metricas: C_v, Topic Diversity, Exclusividade c-TF-IDF, FREX
e estabilidade Jaccard multi-seed.

> **Custoso** (refit completo por combinacao x seed). Habilitar via
> `evaluation.run_outlier_sweep` / `evaluation.run_param_sweep` no
> `params.yaml`. Por padrao ambos sao pulados.


In [ ]:
eval_cfg = params["evaluation"]
top_n_sweep_metrics = eval_cfg.get("top_n_keywords_metrics", 20)
from _helpers import sweep_outlier_strategies, sweep_outlier_threshold, sweep_bertopic_grid


def _sweep_vectorizer():
    """Vectorizer FRESCO por modelo do sweep — update_topics re-fita o
    vectorizer in-place; reusar a instancia global `vectorizer` corromperia
    seu vocab entre variantes."""
    return CountVectorizer(
        stop_words=list(all_stop),
        ngram_range=tuple(bt_cfg["vectorizer"]["ngram_range"]),
        token_pattern=TOKEN_PATTERN, min_df=2,
    )


def _sweep_build_model(seed, n_neighbors=None, min_cluster_size=None, min_samples=None, cluster_selection_method=None):
    nn = n_neighbors if n_neighbors is not None else umap_p["n_neighbors"]
    mcs = min_cluster_size if min_cluster_size is not None else hdbscan_p["min_cluster_size"]
    ms = min_samples if min_samples is not None else hdbscan_p["min_samples"]
    csm = cluster_selection_method if cluster_selection_method is not None else hdbscan_p.get("cluster_selection_method", "leaf")
    umap_s = UMAP(
        n_components=umap_p["n_components"], n_neighbors=nn,
        min_dist=umap_p["min_dist"], metric=umap_p["metric"],
        random_state=seed,
    )
    hdb_s = HDBSCAN(
        min_cluster_size=mcs, min_samples=ms,
        cluster_selection_method=csm,
        prediction_data=True,
    )
    return BERTopic(
        language=BT_LANGUAGE, umap_model=umap_s, hdbscan_model=hdb_s,
        vectorizer_model=_sweep_vectorizer(),
        representation_model=MaximalMarginalRelevance(diversity=bt_cfg["mmr_diversity"]),
        min_topic_size=bt_cfg["min_topic_size"], calculate_probabilities=True, verbose=False,
    )


if eval_cfg.get("run_outlier_sweep", False):
    sweep_seeds = eval_cfg.get("outlier_sweep_seeds", [42, 123, 456, 789, 1024])
    sweep_strategies = eval_cfg.get(
        "outlier_sweep_strategies",
        ["off", "c-tf-idf", "embeddings", "probabilities", "distributions"],
    )
    print(f"Sweep de estrategias ({len(sweep_strategies)} estrategias x {len(sweep_seeds)} seeds)...")
    raw_strat, agg_strat = sweep_outlier_strategies(
        lambda s: _sweep_build_model(s), docs, embeddings, tokenized, dictionary,
        sweep_seeds, strategies=sweep_strategies,
        reduce_nr=bt_cfg.get("reduce_topics_nr"), top_n_metrics=top_n_sweep_metrics,
    )
    raw_strat.to_csv(OUTPUT_DIR / "sweep_outlier_strategies_raw.csv", index=False)
    agg_strat.to_csv(OUTPUT_DIR / "sweep_outlier_strategies.csv", index=False)
    print(agg_strat.to_string(index=False))
else:
    print("Sweep de estrategias desabilitado (evaluation.run_outlier_sweep=false).")
    agg_strat = pd.DataFrame()

if eval_cfg.get("run_threshold_sweep", False):
    thr_seeds = eval_cfg.get("threshold_sweep_seeds", [42])
    thr_grids = eval_cfg.get("threshold_sweep_grids", {})
    if thr_grids:
        n_pts = sum(len(v) for v in thr_grids.values())
        print(f"Sweep de threshold ({len(thr_grids)} estrategias x {n_pts} pontos x {len(thr_seeds)} seeds)...")
        raw_thr, agg_thr = sweep_outlier_threshold(
            lambda s: _sweep_build_model(s), docs, embeddings, tokenized, dictionary,
            thr_seeds, grids=thr_grids,
            reduce_nr=bt_cfg.get("reduce_topics_nr"), top_n_metrics=top_n_sweep_metrics,
        )
        raw_thr.to_csv(OUTPUT_DIR / "sweep_outlier_threshold_raw.csv", index=False)
        agg_thr.to_csv(OUTPUT_DIR / "sweep_outlier_threshold.csv", index=False)
        print(agg_thr.to_string(index=False))
    else:
        print("threshold_sweep_grids vazio — nada a varrer.")
else:
    print("Sweep de threshold desabilitado (evaluation.run_threshold_sweep=false).")
    agg_thr = pd.DataFrame()

if eval_cfg.get("run_param_sweep", False):
    grid_seeds = eval_cfg.get("param_sweep_seeds", [42, 123, 456])
    nn_grid = eval_cfg.get("param_sweep_n_neighbors", [umap_p["n_neighbors"]])
    mcs_grid = eval_cfg.get("param_sweep_min_cluster_size", [hdbscan_p["min_cluster_size"]])
    ms_grid = eval_cfg.get("param_sweep_min_samples", [None])  # [None] = usa o min_samples padrao do corpus
    csm_grid = eval_cfg.get("param_sweep_cluster_selection_methods", ["leaf"])
    print(f"Grid UMAP/HDBSCAN ({len(nn_grid)} nn x {len(mcs_grid)} mcs x {len(ms_grid)} ms x {len(csm_grid)} csm x {len(grid_seeds)} seeds)...")
    raw_grid, agg_grid = sweep_bertopic_grid(
        _sweep_build_model, docs, embeddings, tokenized, dictionary, grid_seeds,
        n_neighbors_grid=nn_grid, min_cluster_size_grid=mcs_grid, min_samples_grid=ms_grid,
        cluster_selection_methods_grid=csm_grid,
        reduce_nr_grid=[bt_cfg.get("reduce_topics_nr")],
        outlier_strategy="off",  # sempre off no grid: isola efeito estrutural
        top_n_metrics=top_n_sweep_metrics,
    )
    raw_grid.to_csv(OUTPUT_DIR / "sweep_bertopic_grid_raw.csv", index=False)
    agg_grid.to_csv(OUTPUT_DIR / "sweep_bertopic_grid.csv", index=False)
    print(agg_grid.to_string(index=False))
else:
    print("Grid UMAP/HDBSCAN desabilitado (evaluation.run_param_sweep=false).")
    agg_grid = pd.DataFrame()


In [ ]:
export_results(
    topics=topics, probs=probs, names=rotulos, texts=docs,
    output_path=str(OUTPUT_DIR / "bertopic_results.csv"),
    topic_type="semantic", granularity="unit", post_ids=post_ids,
)
export_topics_for_eval(
    topics_keywords=topics_keywords_metrics, topics_names=rotulos,
    model_name="bertopic", output_path=str(OUTPUT_DIR / "bertopic_topics_for_eval.csv"),
)

metrics_df = pd.DataFrame([{
    "corpus": corpus_id, "n_topics": len(valid_ids), "outlier_rate": outlier_rate,
    "topic_diversity": td_score, "coherence_cv": cv_score,
    "semantic_coherence": sem_score, "exclusivity": excl_score, "frex": frex_score,
}])
metrics_df.to_csv(OUTPUT_DIR / "bertopic_metrics.csv", index=False)

print(f"Salvo: {OUTPUT_DIR / 'bertopic_results.csv'}  ({len(docs)} docs)")
print(f"Salvo: {OUTPUT_DIR / 'bertopic_topics_for_eval.csv'}")
print(f"Salvo: {OUTPUT_DIR / 'bertopic_metrics.csv'}")

In [ ]:
# Distribuição de docs por tópico
counts = pd.Series(topics).value_counts().sort_index()
counts_valid = counts[counts.index != -1]

plt.figure(figsize=(12, 4))
plt.bar(counts_valid.index, counts_valid.values, color="steelblue")
plt.xlabel("Tópico")
plt.ylabel("Documentos")
plt.title(f"Distribuição de documentos por tópico — {CORPUS}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bertopic_distribution.png", dpi=150)
plt.show()

# Barchart interativo (requer plotly)
try:
    fig = topic_model.visualize_barchart(top_n_topics=min(12, len(valid_ids)), n_words=10)
    fig.show()
except Exception as e:
    print(f"visualize_barchart indisponível: {e}")